# Reproduce the research in the attached reference manuscript, Target Fishing and Molecular Docking of Terpenes. Recapitulate the 23-terpene, four-tool target-fishing consensus design and five-target docking panel (CB2/6KPF, androgen receptor/3V49, CYP19A1/3EQM, estrogen receptor beta/1L2J, and 11-bet

Terpedia reproducibility notebook for study `study_be6fa4a4-c9eb-45e6-916e-aec999992e4f`. This notebook is generated from the immutable research snapshot `4f11d463dff733541dd5275442afa47ff405f2f658e2de2b61227ab9f2859bda` and is compatible with Google Colab.


## Reproducibility contract

The embedded study snapshot is authoritative. Measurements, predictions, failures, and missing results remain distinct. Re-running this notebook verifies recorded provenance and reconstructs analysis tables; external scientific software or private artifacts may require the pinned environments and credentials named in the study record.


In [ ]:
import base64, hashlib, json
from pathlib import Path

EXPECTED_RESEARCH_SHA256 = "4f11d463dff733541dd5275442afa47ff405f2f658e2de2b61227ab9f2859bda"
STUDY = json.loads(base64.b64decode("..."))
print(STUDY["study"]["title"])
print("Study:", STUDY["study"]["id"])
print("Research snapshot:", EXPECTED_RESEARCH_SHA256)


In [ ]:
def canonical(value):
    if isinstance(value, dict):
        return "{" + ",".join(json.dumps(str(k), ensure_ascii=False) + ":" + canonical(value[k]) for k in sorted(value)) + "}"
    if isinstance(value, list):
        return "[" + ",".join(canonical(item) for item in value) + "]"
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))

actual = hashlib.sha256(canonical(STUDY).encode("utf-8")).hexdigest()
assert actual == EXPECTED_RESEARCH_SHA256, (actual, EXPECTED_RESEARCH_SHA256)
print("Research snapshot integrity verified")


## Environment


In [ ]:
import sys, platform
print("Python", sys.version)
print("Platform", platform.platform())
for protocol in STUDY["protocols"]:
    for step in protocol.get("steps", []):
        print(f'{step.get("id")}: {step.get("operation")} | parameters={step.get("parameters", {})}')


## Terpedia natural-language API

The client requests safe JSON-serialized Python data. In Colab, store either a personal JWT as `TERPEDIA_TOKEN` or an approved service key as `TERPEDIA_API_KEY` in Secrets. Credentials are read at runtime and are never written into the notebook.


In [ ]:
import os, requests
from dataclasses import dataclass

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None

def _secret(name):
    if colab_userdata is not None:
        try:
            return colab_userdata.get(name)
        except Exception:
            pass
    return os.environ.get(name)

@dataclass
class TerpediaResult:
    raw: dict

    def __post_init__(self):
        self.variables = self.raw.get("python_data", {}).get("variables", {})
        for name, value in self.variables.items():
            if name.isidentifier() and not hasattr(self, name):
                setattr(self, name, value)

    def frame(self, name="data"):
        value = self.variables.get(name)
        if value is None:
            return pd.DataFrame()
        if isinstance(value, list):
            return pd.json_normalize(value, sep=".")
        if isinstance(value, dict):
            return pd.json_normalize([value], sep=".")
        return pd.DataFrame({name: [value]})

class TerpediaClient:
    def __init__(self, base_url="https://api.terpedia.com", token=None, api_key=None):
        self.base_url = base_url.rstrip("/")
        self.token = token or _secret("TERPEDIA_TOKEN")
        self.api_key = api_key or _secret("TERPEDIA_API_KEY")

    def query(self, question, **options):
        headers = {"Content-Type": "application/json"}
        if self.token:
            headers["Authorization"] = f"Bearer {self.token}"
        elif self.api_key:
            headers["X-API-Key"] = self.api_key
        payload = {"query": question, "response_format": "python_data", **options}
        response = requests.post(f"{self.base_url}/api/v1/query", headers=headers, json=payload, timeout=180)
        response.raise_for_status()
        return TerpediaResult(response.json())

terpedia = TerpediaClient()
research_question = next((p.get("objective") for p in reversed(STUDY["protocols"]) if p.get("objective")), STUDY["study"].get("hypothesis") or STUDY["study"]["title"] )
print("research_question =", research_question)
print("Ready: terpedia_response = terpedia.query(research_question)")
print("Then use terpedia_response.answer, .data, .citations, .variables, or .frame().")


## Study records


In [ ]:
import pandas as pd

runs = pd.json_normalize(STUDY["runs"], sep=".")
evidence = pd.json_normalize(STUDY["evidence"], sep=".")
artifacts = pd.json_normalize(STUDY["artifacts"], sep=".")
display(runs if not runs.empty else pd.DataFrame({"status": ["No runs recorded"]}))
display(evidence if not evidence.empty else pd.DataFrame({"status": ["No evidence recorded"]}))
display(artifacts if not artifacts.empty else pd.DataFrame({"status": ["No artifacts recorded"]}))


## Provenance and artifact verification


In [ ]:
from urllib.request import urlopen

def verify_https_artifact(record):
    uri = record.get("uri", "")
    expected = record.get("sha256")
    if not uri.startswith("https://"):
        return {"uri": uri, "status": "credential_or_protocol_required", "expected_sha256": expected}
    with urlopen(uri) as response:
        actual = hashlib.sha256(response.read()).hexdigest()
    return {"uri": uri, "status": "verified" if actual == expected else "checksum_mismatch", "expected_sha256": expected, "actual_sha256": actual}

artifact_manifest = [{k: item.get(k) for k in ("id", "name", "uri", "sha256", "role")} for item in STUDY["artifacts"]]
display(pd.DataFrame(artifact_manifest))
print("Call verify_https_artifact(record) for public HTTPS artifacts. gs:// artifacts require authenticated Google Cloud access.")


## Recorded results


In [ ]:
numeric_rows = []
for item in STUDY["evidence"]:
    value = item.get("value")
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        numeric_rows.append({
            "subject": item.get("subject", {}).get("label") or item.get("subject", {}).get("id"),
            "predicate": item.get("predicate"), "value": value, "unit": item.get("unit"),
            "kind": item.get("kind"), "status": item.get("evidenceStatus"), "sha256": item.get("sha256")
        })
numeric_evidence = pd.DataFrame(numeric_rows)
display(numeric_evidence if not numeric_evidence.empty else pd.DataFrame({"status": ["No scalar numeric evidence recorded"]}))


In [ ]:
import matplotlib.pyplot as plt
if not runs.empty and "status" in runs:
    runs["status"].value_counts().plot(kind="bar", title="Recorded run outcomes")
    plt.xlabel("Status")
    plt.ylabel("Runs")
    plt.tight_layout()
    plt.show()
if not numeric_evidence.empty:
    numeric_evidence.groupby("predicate")["value"].count().sort_values(ascending=False).head(20).plot(kind="bar", title="Numeric evidence by predicate")
    plt.ylabel("Records")
    plt.tight_layout()
    plt.show()


## Export


In [ ]:
summary = {
    "study_id": STUDY["study"]["id"],
    "research_snapshot_sha256": EXPECTED_RESEARCH_SHA256,
    "protocols": len(STUDY["protocols"]), "runs": len(STUDY["runs"]),
    "evidence_records": len(STUDY["evidence"]), "artifacts": len(STUDY["artifacts"]),
    "run_statuses": runs["status"].value_counts().to_dict() if not runs.empty and "status" in runs else {}
}
Path("terpedia-reproducibility-summary.json").write_text(json.dumps(summary, indent=2))
summary
